In [1]:
#fill NA with None
from sklearn import tree
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import random
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
from statistics import mean
from sklearn import svm

In [2]:
#https://stackoverflow.com/questions/20224526/how-to-extract-the-decision-rules-from-scikit-learn-decision-tree
def get_code(tree, feature_names):
        left      = tree.tree_.children_left
        right     = tree.tree_.children_right
        threshold = tree.tree_.threshold
        features  = [feature_names[i] for i in tree.tree_.feature]
        value = tree.tree_.value

        def recurse(left, right, threshold, features, node):
                if (threshold[node] != -2):
                        print("if ( " + features[node] + " <= " + str(threshold[node]) + " ) {")
                        if left[node] != -1:
                                recurse (left, right, threshold, features,left[node])
                        print( "} else {")
                        if right[node] != -1:
                                recurse (left, right, threshold, features,right[node])
                        print( "}")
                else:
                        print( "return " + str(value[node]))

        recurse(left, right, threshold, features, 0)

In [3]:
data = pd.read_excel('dataset_gen.xlsx') 
dataset_all_gen = pd.read_excel('dataset_all_gen.xlsx')
dataset_all_original = pd.read_excel('dataset_all_original.xlsx')
dataset_jaccard_gen = pd.read_excel('dataset_jaccard_gen.xlsx')
dataset_jaccard_original = pd.read_excel('dataset_jaccard_original.xlsx')
dataset_myroom_gen = pd.read_excel('dataset_myroom_gen.xlsx')
dataset_myroom_original = pd.read_excel('dataset_myroom_original.xlsx')
dataset_university_gen = pd.read_excel('dataset_university_gen.xlsx')
dataset_university_original = pd.read_excel('dataset_university_original.xlsx')

In [4]:
data = data.sample(frac = 1) 
dataset_all_gen = dataset_all_gen.sample(frac = 1) 
dataset_all_original = dataset_all_original.sample(frac = 1) 
dataset_jaccard_gen = dataset_jaccard_gen.sample(frac = 1) 
dataset_jaccard_original = dataset_jaccard_original.sample(frac = 1) 
dataset_myroom_gen = dataset_myroom_gen.sample(frac = 1) 
dataset_myroom_original = dataset_myroom_original.sample(frac = 1) 
dataset_university_gen = dataset_university_gen.sample(frac = 1) 
dataset_university_original = dataset_university_original.sample(frac = 1) 

In [5]:

data = data.reset_index(drop=True)
dataset_all_gen = dataset_all_gen.reset_index(drop=True)
dataset_all_original = dataset_all_original.reset_index(drop=True)
dataset_jaccard_gen = dataset_jaccard_gen.reset_index(drop=True)
dataset_jaccard_original = dataset_jaccard_original.reset_index(drop=True)
dataset_myroom_gen = dataset_myroom_gen.reset_index(drop=True)
dataset_myroom_original = dataset_myroom_original.reset_index(drop=True)
dataset_university_gen = dataset_university_gen.reset_index(drop=True)
dataset_university_original = dataset_university_original.reset_index(drop=True)

In [6]:
data.isnull().sum()

type             0
Name             0
Vendor           0
DNS           2304
mDNS          2704
NUPNP        11000
open_port     1731
SSDP_Type     2304
SSDP_t       11000
SSDP_v       11000
dtype: int64

In [7]:
data = data.fillna("NaN")

In [8]:
#fill Na
dataset_all_gen = dataset_all_gen.fillna("NaN")
dataset_all_original = dataset_all_original.fillna("NaN")
dataset_jaccard_gen = dataset_jaccard_gen.fillna("NaN")
dataset_jaccard_original = dataset_jaccard_original.fillna("NaN")
dataset_myroom_gen = dataset_myroom_gen.fillna("NaN")
dataset_myroom_original = dataset_myroom_original.fillna("NaN")
dataset_university_gen = dataset_university_gen.fillna("NaN")
dataset_university_original = dataset_university_original.fillna("NaN")

In [9]:
data.isnull().sum()

type         0
Name         0
Vendor       0
DNS          0
mDNS         0
NUPNP        0
open_port    0
SSDP_Type    0
SSDP_t       0
SSDP_v       0
dtype: int64

In [10]:
data.dtypes

type         object
Name         object
Vendor       object
DNS          object
mDNS         object
NUPNP        object
open_port    object
SSDP_Type    object
SSDP_t       object
SSDP_v       object
dtype: object

In [11]:
data = data.astype('category')

In [12]:
dataset_all_gen = dataset_all_gen.astype('category')
dataset_all_original = dataset_all_original.astype('category')
dataset_jaccard_gen = dataset_jaccard_gen.astype('category')
dataset_jaccard_original = dataset_jaccard_original.astype('category')
dataset_myroom_gen = dataset_myroom_gen.astype('category')
dataset_myroom_original = dataset_myroom_original.astype('category')
dataset_university_gen = dataset_university_gen.astype('category')
dataset_university_original = dataset_university_original.astype('category')

In [13]:
data.dtypes

type         category
Name         category
Vendor       category
DNS          category
mDNS         category
NUPNP        category
open_port    category
SSDP_Type    category
SSDP_t       category
SSDP_v       category
dtype: object

In [14]:
data

,type,Name,Vendor,DNS,mDNS,NUPNP,open_port,SSDP_Type,SSDP_t,SSDP_v
0,Energy Management,LiFX Smart Bulb,LIFI LABS MANAGEMENT PTY LTD,state.tx.us,_fintone._udp.local.,NaN,NaN,NOTIFY * HTTP/1.1,NaN,NaN
1,Cameras,Wyze Cam,Wyze Labs,NaN,_wrapsafe._udp.local.,NaN,tcp/1883,NaN,NaN,NaN
2,Cameras,Dropcam,Dropcam,etsy.com,NaN,NaN,tcp/22,NOTIFY * HTTP/1.1,NaN,NaN
3,Non-IOT,MacBook4,"Apple, Inc.",NaN,_matsoft._udp.local.,NaN,tcp/80,NaN,NaN,NaN
4,Appliances,Smart Things,"SmartThings, Inc",shinystat.com,_voyatouch._tcp.local.,NaN,tcp/443,NOTIFY * HTTP/1.1,NaN,NaN
5,Non-IOT,Asus ZenPad Tablet,ASUSTek COMPUTER INC.,cnet.com,NaN,NaN,tcp/443,M-SEARCH * HTTP/1.1,NaN,NaN
6,Hubs,Google OnHub,Google,i2i.jp,NaN,NaN,tcp/1900,NOTIFY * HTTP/1.1,NaN,NaN
7,Appliances,iRobot,iRobot Corporation,icio.us,NaN,NaN,tcp/1883,NOTIFY * HTTP/1.1,NaN,NaN
8,Cameras,D-Link DCS-5009L Camera,D‑Link Ltd,t.co,_fix san._tcp.local.,NaN,NaN,NOTIFY * HTTP/1.1,NaN,NaN
9,Cameras,Nest Dropcam,Nest Labs Inc.,surveymonkey.com,_daltfresh._udp.local.,NaN,tcp/80,NOTIFY * HTTP/1.1,NaN,NaN


In [15]:
y = data.drop_duplicates(keep='first')
y = y.drop(['Name'], axis=1)
y = y.drop_duplicates(keep='first')
# y = y.reset_index()
# shuffle the DataFrame rows
# y = y.sample(frac = 1) 


In [16]:
len(y)

10610

In [17]:
#drop_duplicates
dataset_all_gen_y = dataset_all_gen.drop_duplicates(keep='first')
dataset_all_original_y = dataset_all_original.drop_duplicates(keep='first')
dataset_jaccard_gen_y = dataset_jaccard_gen.drop_duplicates(keep='first')
dataset_jaccard_original_y = dataset_jaccard_original.drop_duplicates(keep='first')
dataset_myroom_gen_y = dataset_myroom_gen.drop_duplicates(keep='first')
dataset_myroom_original_y = dataset_myroom_original.drop_duplicates(keep='first')
dataset_university_gen_y = dataset_university_gen.drop_duplicates(keep='first')
dataset_university_original_y = dataset_university_original.drop_duplicates(keep='first')

In [18]:
#drop name
dataset_all_gen_y = dataset_all_gen_y.drop(['Name'], axis=1)
dataset_all_original_y = dataset_all_original_y.drop(['Name'], axis=1)
dataset_jaccard_gen_y = dataset_jaccard_gen_y.drop(['Name'], axis=1)
dataset_jaccard_original_y = dataset_jaccard_original_y.drop(['Name'], axis=1)
dataset_myroom_gen_y = dataset_myroom_gen_y.drop(['Name'], axis=1)
dataset_myroom_original_y = dataset_myroom_original_y.drop(['Name'], axis=1)
dataset_university_gen_y = dataset_university_gen_y.drop(['Name'], axis=1)
dataset_university_original_y = dataset_university_original_y.drop(['Name'], axis=1)

In [19]:
#drop_duplicates
dataset_all_gen_y = dataset_all_gen_y.drop_duplicates(keep='first')
dataset_all_original_y = dataset_all_original_y.drop_duplicates(keep='first')
dataset_jaccard_gen_y = dataset_jaccard_gen_y.drop_duplicates(keep='first')
dataset_jaccard_original_y = dataset_jaccard_original_y.drop_duplicates(keep='first')
dataset_myroom_gen_y = dataset_myroom_gen_y.drop_duplicates(keep='first')
dataset_myroom_original_y = dataset_myroom_original_y.drop_duplicates(keep='first')
dataset_university_gen_y = dataset_university_gen_y.drop_duplicates(keep='first')
dataset_university_original_y = dataset_university_original_y.drop_duplicates(keep='first')

In [20]:
y = y.apply(LabelEncoder().fit_transform)

In [21]:
dataset_all_gen_y = dataset_all_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_all_original_y = dataset_all_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_jaccard_gen_y = dataset_jaccard_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_jaccard_original_y = dataset_jaccard_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_myroom_gen_y = dataset_myroom_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_myroom_original_y = dataset_myroom_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_university_gen_y = dataset_university_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_university_original_y = dataset_university_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')

In [22]:
list_data = []
# list_data.append(y)
# list_data.append(dataset_all_gen_y)
list_data.append(dataset_all_original_y)
# list_data.append(dataset_jaccard_gen_y)
list_data.append(dataset_jaccard_original_y)
# list_data.append(dataset_myroom_gen_y)
list_data.append(dataset_myroom_original_y)
# list_data.append(dataset_university_gen_y)
list_data.append(dataset_university_original_y)
len(list_data)

4

In [23]:
accuracy = []
accuracy_t = []
accuracy_temp = []

In [24]:
for data in list_data:
    for i in range(100):
        kf = KFold(n_splits=10, random_state=None, shuffle=True)
#         kf.get_n_splits(data)
        for train_index, test_index in kf.split(data):
        #     print(index)
#             print("TRAIN:", train_index, "TEST:", test_index)
            train, test = data.iloc[train_index], data.iloc[test_index]
#             print("TRAIN:", train, "TEST:", test)
        #     print(len(train))
#             print("TRAIN:", len(train), "TEST:", len(test))
            label_data = train.type
            train_data = train.drop(['type'], axis=1)
            tlabel_data = test.type
            test_data = test.drop(['type'], axis=1)
            truelabel_data = pd.DataFrame(tlabel_data)
            truelabel_data = truelabel_data.reset_index(drop=True)
            clf = clf = svm.SVC(decision_function_shape='ovr')
            clf = clf.fit(train_data,label_data )
            predict = clf.predict(test_data)
#             print(truelabel_data)
#             print(predict)
        #     len(predict)
            result_label_data = pd.DataFrame(columns = ['type'])
            result_label_data.type = predict
            compare = pd.DataFrame(columns = ['truelabel_data','predictedlabel'])
            compare.truelabel_data = truelabel_data.type
            compare.predictedlabel = result_label_data.type
#             print(compare)
            t = (accuracy_score(compare.truelabel_data, compare.predictedlabel))
            accuracy_temp.append(t)
            print(i,"  Accuracy: ",t)
        accuracy_t.append(mean(accuracy_temp))
    accuracy.append(mean(accuracy_t))

0   Accuracy:  0.6323529411764706
0   Accuracy:  0.5735294117647058
0   Accuracy:  0.6176470588235294
0   Accuracy:  0.6029411764705882
0   Accuracy:  0.5441176470588235
0   Accuracy:  0.5588235294117647
0   Accuracy:  0.6911764705882353
0   Accuracy:  0.5223880597014925
0   Accuracy:  0.6417910447761194
0   Accuracy:  0.6716417910447762
1   Accuracy:  0.6764705882352942
1   Accuracy:  0.6323529411764706
1   Accuracy:  0.6029411764705882
1   Accuracy:  0.6176470588235294
1   Accuracy:  0.6470588235294118
1   Accuracy:  0.6029411764705882
1   Accuracy:  0.5441176470588235
1   Accuracy:  0.6417910447761194
1   Accuracy:  0.5074626865671642
1   Accuracy:  0.5522388059701493
2   Accuracy:  0.6323529411764706
2   Accuracy:  0.5294117647058824
2   Accuracy:  0.6911764705882353
2   Accuracy:  0.6911764705882353
2   Accuracy:  0.5588235294117647
2   Accuracy:  0.6470588235294118
2   Accuracy:  0.6029411764705882
2   Accuracy:  0.582089552238806
2   Accuracy:  0.5970149253731343
2   Accuracy:  

24   Accuracy:  0.6865671641791045
24   Accuracy:  0.5522388059701493
24   Accuracy:  0.5373134328358209
25   Accuracy:  0.6176470588235294
25   Accuracy:  0.6029411764705882
25   Accuracy:  0.6029411764705882
25   Accuracy:  0.6764705882352942
25   Accuracy:  0.5147058823529411
25   Accuracy:  0.5735294117647058
25   Accuracy:  0.5147058823529411
25   Accuracy:  0.746268656716418
25   Accuracy:  0.6268656716417911
25   Accuracy:  0.5671641791044776
26   Accuracy:  0.5735294117647058
26   Accuracy:  0.6764705882352942
26   Accuracy:  0.5147058823529411
26   Accuracy:  0.6764705882352942
26   Accuracy:  0.6323529411764706
26   Accuracy:  0.6323529411764706
26   Accuracy:  0.75
26   Accuracy:  0.5671641791044776
26   Accuracy:  0.47761194029850745
26   Accuracy:  0.5522388059701493
27   Accuracy:  0.5588235294117647
27   Accuracy:  0.6029411764705882
27   Accuracy:  0.6029411764705882
27   Accuracy:  0.5441176470588235
27   Accuracy:  0.7205882352941176
27   Accuracy:  0.6470588235294118

49   Accuracy:  0.6323529411764706
49   Accuracy:  0.6470588235294118
49   Accuracy:  0.75
49   Accuracy:  0.5735294117647058
49   Accuracy:  0.6764705882352942
49   Accuracy:  0.5074626865671642
49   Accuracy:  0.5671641791044776
49   Accuracy:  0.7164179104477612
50   Accuracy:  0.6029411764705882
50   Accuracy:  0.6470588235294118
50   Accuracy:  0.5588235294117647
50   Accuracy:  0.6617647058823529
50   Accuracy:  0.5441176470588235
50   Accuracy:  0.5882352941176471
50   Accuracy:  0.6323529411764706
50   Accuracy:  0.582089552238806
50   Accuracy:  0.5373134328358209
50   Accuracy:  0.7164179104477612
51   Accuracy:  0.6764705882352942
51   Accuracy:  0.6323529411764706
51   Accuracy:  0.5882352941176471
51   Accuracy:  0.6029411764705882
51   Accuracy:  0.5735294117647058
51   Accuracy:  0.6470588235294118
51   Accuracy:  0.5588235294117647
51   Accuracy:  0.6716417910447762
51   Accuracy:  0.5373134328358209
51   Accuracy:  0.5671641791044776
52   Accuracy:  0.6029411764705882


73   Accuracy:  0.6764705882352942
73   Accuracy:  0.5373134328358209
73   Accuracy:  0.6716417910447762
73   Accuracy:  0.5970149253731343
74   Accuracy:  0.6323529411764706
74   Accuracy:  0.5441176470588235
74   Accuracy:  0.5441176470588235
74   Accuracy:  0.6323529411764706
74   Accuracy:  0.6323529411764706
74   Accuracy:  0.5735294117647058
74   Accuracy:  0.6323529411764706
74   Accuracy:  0.6119402985074627
74   Accuracy:  0.6716417910447762
74   Accuracy:  0.5671641791044776
75   Accuracy:  0.6323529411764706
75   Accuracy:  0.5882352941176471
75   Accuracy:  0.7058823529411765
75   Accuracy:  0.5294117647058824
75   Accuracy:  0.5735294117647058
75   Accuracy:  0.5147058823529411
75   Accuracy:  0.6911764705882353
75   Accuracy:  0.5671641791044776
75   Accuracy:  0.6567164179104478
75   Accuracy:  0.5970149253731343
76   Accuracy:  0.6617647058823529
76   Accuracy:  0.6323529411764706
76   Accuracy:  0.6176470588235294
76   Accuracy:  0.7058823529411765
76   Accuracy:  0.51

97   Accuracy:  0.6470588235294118
97   Accuracy:  0.6470588235294118
97   Accuracy:  0.5223880597014925
97   Accuracy:  0.6417910447761194
97   Accuracy:  0.5671641791044776
98   Accuracy:  0.6176470588235294
98   Accuracy:  0.6323529411764706
98   Accuracy:  0.47058823529411764
98   Accuracy:  0.6617647058823529
98   Accuracy:  0.6470588235294118
98   Accuracy:  0.5588235294117647
98   Accuracy:  0.5735294117647058
98   Accuracy:  0.6119402985074627
98   Accuracy:  0.7164179104477612
98   Accuracy:  0.5970149253731343
99   Accuracy:  0.6176470588235294
99   Accuracy:  0.6176470588235294
99   Accuracy:  0.6323529411764706
99   Accuracy:  0.5294117647058824
99   Accuracy:  0.6764705882352942
99   Accuracy:  0.5441176470588235
99   Accuracy:  0.5735294117647058
99   Accuracy:  0.6268656716417911
99   Accuracy:  0.7014925373134329
99   Accuracy:  0.5671641791044776
0   Accuracy:  0.8709677419354839
0   Accuracy:  0.967741935483871
0   Accuracy:  0.8333333333333334
0   Accuracy:  1.0
0   

24   Accuracy:  0.8666666666666667
24   Accuracy:  0.9333333333333333
24   Accuracy:  0.9333333333333333
24   Accuracy:  0.9666666666666667
24   Accuracy:  0.9
24   Accuracy:  0.8666666666666667
25   Accuracy:  0.967741935483871
25   Accuracy:  0.967741935483871
25   Accuracy:  0.8333333333333334
25   Accuracy:  0.9
25   Accuracy:  0.9
25   Accuracy:  0.8666666666666667
25   Accuracy:  0.9333333333333333
25   Accuracy:  0.9
25   Accuracy:  0.9333333333333333
25   Accuracy:  0.9
26   Accuracy:  0.8387096774193549
26   Accuracy:  0.9354838709677419
26   Accuracy:  0.8333333333333334
26   Accuracy:  0.9333333333333333
26   Accuracy:  0.9333333333333333
26   Accuracy:  0.9666666666666667
26   Accuracy:  0.8666666666666667
26   Accuracy:  1.0
26   Accuracy:  0.8666666666666667
26   Accuracy:  0.9
27   Accuracy:  0.9032258064516129
27   Accuracy:  0.8709677419354839
27   Accuracy:  0.8666666666666667
27   Accuracy:  0.9
27   Accuracy:  0.9333333333333333
27   Accuracy:  0.9666666666666667
27

... (6 outputs left out)

9   Accuracy:  0.5862068965517241
9   Accuracy:  0.42857142857142855
9   Accuracy:  0.42857142857142855
9   Accuracy:  0.5
9   Accuracy:  0.4642857142857143
9   Accuracy:  0.5357142857142857
9   Accuracy:  0.35714285714285715
10   Accuracy:  0.41379310344827586
10   Accuracy:  0.5172413793103449
10   Accuracy:  0.3793103448275862
10   Accuracy:  0.5172413793103449
10   Accuracy:  0.4642857142857143
10   Accuracy:  0.35714285714285715
10   Accuracy:  0.39285714285714285
10   Accuracy:  0.4642857142857143
10   Accuracy:  0.6428571428571429
10   Accuracy:  0.39285714285714285
11   Accuracy:  0.5172413793103449
11   Accuracy:  0.5517241379310345
11   Accuracy:  0.3793103448275862
11   Accuracy:  0.3793103448275862
11   Accuracy:  0.5
11   Accuracy:  0.42857142857142855
11   Accuracy:  0.35714285714285715
11   Accuracy:  0.39285714285714285
11   Accuracy:  0.4642857142857143
11   Accuracy:  0.35714285714285715
12   Accuracy:  0.4482758620689655
12   Accuracy:  0.5517241379310345
12   Accura

34   Accuracy:  0.4827586206896552
34   Accuracy:  0.4482758620689655
34   Accuracy:  0.5714285714285714
34   Accuracy:  0.39285714285714285
34   Accuracy:  0.5714285714285714
34   Accuracy:  0.4642857142857143
34   Accuracy:  0.4642857142857143
34   Accuracy:  0.35714285714285715
35   Accuracy:  0.6551724137931034
35   Accuracy:  0.3448275862068966
35   Accuracy:  0.41379310344827586
35   Accuracy:  0.4482758620689655
35   Accuracy:  0.4642857142857143
35   Accuracy:  0.39285714285714285
35   Accuracy:  0.39285714285714285
35   Accuracy:  0.6071428571428571
35   Accuracy:  0.4642857142857143
35   Accuracy:  0.32142857142857145
36   Accuracy:  0.4827586206896552
36   Accuracy:  0.41379310344827586
36   Accuracy:  0.4482758620689655
36   Accuracy:  0.41379310344827586
36   Accuracy:  0.5357142857142857
36   Accuracy:  0.35714285714285715
36   Accuracy:  0.5
36   Accuracy:  0.35714285714285715
36   Accuracy:  0.5
36   Accuracy:  0.4642857142857143
37   Accuracy:  0.4827586206896552
37   

58   Accuracy:  0.4642857142857143
58   Accuracy:  0.35714285714285715
58   Accuracy:  0.39285714285714285
58   Accuracy:  0.42857142857142855
58   Accuracy:  0.42857142857142855
59   Accuracy:  0.5517241379310345
59   Accuracy:  0.4827586206896552
59   Accuracy:  0.27586206896551724
59   Accuracy:  0.41379310344827586
59   Accuracy:  0.42857142857142855
59   Accuracy:  0.42857142857142855
59   Accuracy:  0.4642857142857143
59   Accuracy:  0.5
59   Accuracy:  0.5357142857142857
59   Accuracy:  0.42857142857142855
60   Accuracy:  0.4482758620689655
60   Accuracy:  0.41379310344827586
60   Accuracy:  0.4482758620689655
60   Accuracy:  0.5862068965517241
60   Accuracy:  0.5
60   Accuracy:  0.21428571428571427
60   Accuracy:  0.5357142857142857
60   Accuracy:  0.42857142857142855
60   Accuracy:  0.2857142857142857
60   Accuracy:  0.5714285714285714
61   Accuracy:  0.5517241379310345
61   Accuracy:  0.5172413793103449
61   Accuracy:  0.4827586206896552
61   Accuracy:  0.4482758620689655
61 

83   Accuracy:  0.42857142857142855
84   Accuracy:  0.41379310344827586
84   Accuracy:  0.5172413793103449
84   Accuracy:  0.4827586206896552
84   Accuracy:  0.3793103448275862
84   Accuracy:  0.4642857142857143
84   Accuracy:  0.4642857142857143
84   Accuracy:  0.5
84   Accuracy:  0.5
84   Accuracy:  0.42857142857142855
84   Accuracy:  0.39285714285714285
85   Accuracy:  0.3793103448275862
85   Accuracy:  0.5172413793103449
85   Accuracy:  0.4827586206896552
85   Accuracy:  0.3103448275862069
85   Accuracy:  0.39285714285714285
85   Accuracy:  0.5
85   Accuracy:  0.39285714285714285
85   Accuracy:  0.35714285714285715
85   Accuracy:  0.6785714285714286
85   Accuracy:  0.42857142857142855
86   Accuracy:  0.5172413793103449
86   Accuracy:  0.41379310344827586
86   Accuracy:  0.3448275862068966
86   Accuracy:  0.3793103448275862
86   Accuracy:  0.39285714285714285
86   Accuracy:  0.4642857142857143
86   Accuracy:  0.42857142857142855
86   Accuracy:  0.5
86   Accuracy:  0.5
86   Accuracy:

In [25]:
# print(len(y),"\t Accuracy for dataset_gen: \t", accuracy[0])
# print(len(dataset_all_gen_y),"\t Accuracy for dataset_all_gen: \t", accuracy[1])
print(len(dataset_all_original_y),"\t Accuracy for dataset_all_original: \t", accuracy[0])
# print(len(dataset_all_original_y),"\t Accuracy for dataset_all_original: \t", accuracy[2])
# print(len(dataset_jaccard_gen_y),"\t Accuracy for dataset_jaccard_gen: \t", accuracy[3])
print(len(dataset_jaccard_original_y),"\t Accuracy for dataset_jaccard_original: \t", accuracy[1])
# print(len(dataset_jaccard_original_y),"\t Accuracy for dataset_jaccard_original: \t", accuracy[4])
# print(len(dataset_myroom_gen_y),"\t Accuracy for dataset_myroom_gen: \t", accuracy[5])
print(len(dataset_myroom_original_y),"\t Accuracy for dataset_myroom_original: \t", accuracy[2])
# print(len(dataset_myroom_original_y),"\t Accuracy for dataset_myroom_original: \t", accuracy[6])
# print(len(dataset_university_gen_y),"\t Accuracy for dataset_university_gen: \t", accuracy[7])
print(len(dataset_university_original_y),"\t Accuracy for dataset_university_original: \t", accuracy[3])
# print(len(dataset_university_original_y),"\t Accuracy for dataset_university_original: \t", accuracy[8])

677 	 Accuracy for dataset_all_original: 	 0.6055066560931585
302 	 Accuracy for dataset_jaccard_original: 	 0.6522920721873425
91 	 Accuracy for dataset_myroom_original: 	 0.687478945725787
284 	 Accuracy for dataset_university_original: 	 0.694377788628292
